# Hinglish ASR

## 1. Download the dataset

I use the **Hindi-English code-switched** training split from [OpenSLR 104](https://www.openslr.org/104/) (MUCS 2021).

The archive (~7.3 GB) is downloaded and extracted into `data/raw/hinglish_openslr_raw/`. The download is resumable: if it is interrupted, re-running the cells continues from where it stopped. The `.tar.gz` is deleted after extraction to save disk space.

In [1]:
import shutil
import tarfile
import urllib.request
import wave
from pathlib import Path

### Paths

In [2]:
# EU mirror of https://www.openslr.org/resources/104/ (same file, usually much faster than the main server)
DATA_URL = "https://openslr.elda.org/resources/104/Hindi-English_train.tar.gz"
CHUNK_SIZE = 1024 * 1024  # 1 MiB per read/write

RAW_DIR = Path.cwd().parent / "data" / "raw"
ARCHIVE_PATH = RAW_DIR / "Hindi-English_train.tar.gz"
PART_PATH = RAW_DIR / "Hindi-English_train.tar.gz.part"  # in-progress download
DATASET_DIR = RAW_DIR / "hinglish_openslr_raw"
TMP_DIR = RAW_DIR / "hinglish_openslr_raw.tmp"  # in-progress extraction

RAW_DIR.mkdir(parents=True, exist_ok=True)

### Resumable download

Bytes are appended to a `.part` file as they arrive. On a re-run, an HTTP `Range` request asks the server only for the bytes not yet on disk (OpenSLR supports this). The `.part` file is renamed to the final archive name only once its size matches the server's, so a file named `.tar.gz` is always complete.

In [3]:
def download(url: str, part_path: Path, dest_path: Path) -> None:
    """Download `url` to `dest_path`, resuming from `part_path` if a partial download exists."""
    # Total size of the file on the server
    with urllib.request.urlopen(urllib.request.Request(url, method="HEAD")) as response:
        total = int(response.headers["Content-Length"])

    # Bytes already downloaded by a previous, interrupted run
    start = part_path.stat().st_size if part_path.exists() else 0

    if start < total:
        request = urllib.request.Request(url, headers={"Range": f"bytes={start}-"})
        with (
            urllib.request.urlopen(request) as response,
            part_path.open("ab") as f,
            tqdm(total=total, initial=start, unit="B", unit_scale=True, unit_divisor=1024, desc=dest_path.name) as pbar,
        ):
            # 206 means the server is sending only the requested range; anything else would corrupt the appended file
            if response.status != 206:
                raise RuntimeError(f"Server did not honour the Range request (HTTP {response.status})")
            while chunk := response.read(CHUNK_SIZE):
                f.write(chunk)
                pbar.update(len(chunk))

    size = part_path.stat().st_size
    if size != total:
        raise RuntimeError(f"Incomplete download ({size}/{total} bytes); re-run the cell to resume")
    part_path.rename(dest_path)

### Download and extract

- Skipped entirely if the dataset has already been extracted.
- Download is skipped if the complete archive is already on disk.
- Gzip can't resume mid-stream, so an interrupted extraction restarts from scratch (from the local archive, no re-download). It extracts into a temporary folder that is renamed only on success, so a half-extracted dataset is never mistaken for a complete one.

In [4]:
if not DATASET_DIR.exists():
    if not ARCHIVE_PATH.exists():
        download(DATA_URL, PART_PATH, ARCHIVE_PATH)

    # Clear leftovers from an interrupted extraction, then extract into the temporary folder.
    # filter="data" blocks unsafe members (absolute paths, links outside the target).
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(TMP_DIR, filter="data")

    # Extraction succeeded: publish the dataset and remove the archive
    TMP_DIR.rename(DATASET_DIR)
    ARCHIVE_PATH.unlink()

# Quick sanity check of the extracted contents
sorted(p.name for p in DATASET_DIR.iterdir())

['train']